# Modelagem e Avaliação
## Treino, Validação e Comparação dos Modelos

**Objetivo:** Treinar 3 modelos de regressão para prever a média de gols por jogo de cada seleção na Copa do Mundo, avaliar seu desempenho com validação cruzada temporal e comparar estatisticamente os resultados.

**Divisão temporal:**
- **Treino:** Copas de 1994 a 2018 (216 amostras)
- **Teste:** Copa 2022 (32 amostras)
- **Previsão final:** Copa 2026 (sem target disponível)

**Métrica primária:** MAE (Mean Absolute Error) — interpretável em gols por jogo  
**Métrica secundária:** RMSE — penaliza erros grandes

**Dataset:** `data/processed/features_completo.csv`

## 1. Imports e Carregamento dos Dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from xgboost import XGBRegressor
from scipy.stats import wilcoxon

sns.set_theme(style='whitegrid')
np.random.seed(42)

df = pd.read_csv('../data/processed/features_completo.csv')

print(f'Shape: {df.shape}')
df.head()

## 2. Separação de Features e Target

As colunas `selecao` e `copa_alvo` são usadas para identificação e divisão temporal, mas **não entram como features** do modelo.

A divisão treino/teste respeita a ordem temporal — nunca usamos dados do futuro para treinar.

In [ ]:
features = [
    'media_gols_marcados_ciclo',
    'media_gols_sofridos_ciclo',
    'pct_vitorias_ciclo',
    'total_jogos_ciclo',
    'media_gols_marcados_ult15',
    'media_gols_sofridos_ult15',
    'pct_vitorias_ult15'
]

X = df[features]
y = df['media_gols_copa']

# Divisão temporal
X_train = X[df['copa_alvo'] < 2022]
X_test  = X[df['copa_alvo'] == 2022]
y_train = y[df['copa_alvo'] < 2022]
y_test  = y[df['copa_alvo'] == 2022]

print(f'Treino: {X_train.shape[0]} amostras — Copas {sorted(df[df["copa_alvo"] < 2022]["copa_alvo"].unique())}')
print(f'Teste:  {X_test.shape[0]} amostras — Copa 2022')
print(f'\nDistribuição do target (treino):')
print(y_train.describe().round(3))

## 3. Treino dos Modelos

Três modelos foram escolhidos para cobrir diferentes abordagens:

| Modelo | Justificativa |
|--------|---------------|
| **Regressão Linear** | Baseline interpretável — verifica se relação linear é suficiente |
| **Random Forest** | Captura não-linearidades; robusto a outliers |
| **XGBoost** | Estado da arte para dados tabulares estruturados |

O `random_state=42` garante reprodutibilidade dos resultados (RQ-RP-02).

In [ ]:
# Regressão Linear
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

# Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

# XGBoost
xgb = XGBRegressor(n_estimators=100, random_state=42)
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)

print('Modelos treinados com sucesso!')

## 4. Avaliação no Conjunto de Teste (Copa 2022)

O conjunto de teste é usado **apenas uma vez**, na avaliação final — nunca para ajustar hiperparâmetros (RQ-PM-04).

In [ ]:
mae_lr   = mean_absolute_error(y_test, y_pred_lr)
rmse_lr  = root_mean_squared_error(y_test, y_pred_lr)

mae_rf   = mean_absolute_error(y_test, y_pred_rf)
rmse_rf  = root_mean_squared_error(y_test, y_pred_rf)

mae_xgb  = mean_absolute_error(y_test, y_pred_xgb)
rmse_xgb = root_mean_squared_error(y_test, y_pred_xgb)

resultados = pd.DataFrame({
    'Modelo':  ['Regressão Linear', 'Random Forest', 'XGBoost'],
    'MAE':     [mae_lr, mae_rf, mae_xgb],
    'RMSE':    [rmse_lr, rmse_rf, rmse_xgb]
}).sort_values('MAE').reset_index(drop=True)

print('Resultados no conjunto de teste (Copa 2022):')
print(resultados.to_string(index=False))

## 5. Visualização — Previsão vs. Real (Copa 2022)

Pontos próximos à linha vermelha indicam boa previsão. Pontos acima da linha = modelo superestimou; abaixo = subestimou.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

modelos_viz = [
    ('Regressão Linear', y_pred_lr, mae_lr),
    ('Random Forest',    y_pred_rf, mae_rf),
    ('XGBoost',          y_pred_xgb, mae_xgb)
]

for ax, (nome, y_pred, mae) in zip(axes, modelos_viz):
    ax.scatter(y_test, y_pred, alpha=0.7, color='steelblue', edgecolors='black')
    ax.plot([0, 3], [0, 3], 'r--', label='Previsão perfeita')
    ax.set_xlabel('Gols Reais')
    ax.set_ylabel('Gols Previstos')
    ax.set_title(f'{nome}\nMAE = {mae:.4f}')
    ax.legend()

plt.suptitle('Previsão vs. Real — Copa 2022', fontsize=14)
plt.tight_layout()
plt.savefig('../article/figures/modelos_comparacao.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Validação Cruzada Temporal (Walk-Forward)

A validação cruzada simples (k-fold aleatório) não funciona para dados temporais — usaríamos dados do futuro para prever o passado. O método correto é o **walk-forward**: treinamos em Copas antigas e testamos na Copa seguinte, avançando uma Copa por vez.

```
Fold 1: Treino [1994]         → Teste [1998]
Fold 2: Treino [1994, 1998]   → Teste [2002]
Fold 3: Treino [1994–2002]    → Teste [2006]
...
Fold 6: Treino [1994–2014]    → Teste [2018]
```

Isso produz a **média e desvio-padrão do MAE** exigidos pelo RQ-RP-02.

In [ ]:
copas_ordenadas = sorted(df['copa_alvo'].unique())

# Walk-forward: mínimo 2 Copas no treino para ter dados suficientes
folds = [
    (copas_ordenadas[:i], copas_ordenadas[i])
    for i in range(2, len(copas_ordenadas))  # exclui Copa 2022 (usada só no teste final)
]

mae_folds = {'Regressão Linear': [], 'Random Forest': [], 'XGBoost': []}

for copas_treino, copa_teste in folds:
    mask_treino = df['copa_alvo'].isin(copas_treino)
    mask_teste  = df['copa_alvo'] == copa_teste

    Xtr, ytr = X[mask_treino], y[mask_treino]
    Xte, yte = X[mask_teste],  y[mask_teste]

    # Regressão Linear
    m_lr = LinearRegression().fit(Xtr, ytr)
    mae_folds['Regressão Linear'].append(mean_absolute_error(yte, m_lr.predict(Xte)))

    # Random Forest
    m_rf = RandomForestRegressor(n_estimators=100, random_state=42).fit(Xtr, ytr)
    mae_folds['Random Forest'].append(mean_absolute_error(yte, m_rf.predict(Xte)))

    # XGBoost
    m_xgb = XGBRegressor(n_estimators=100, random_state=42).fit(Xtr, ytr)
    mae_folds['XGBoost'].append(mean_absolute_error(yte, m_xgb.predict(Xte)))

print('Validação Cruzada Temporal (Walk-Forward):')
print(f'{"Modelo":<20} {"MAE Médio":>10} {"Desvio-Padrão":>15}')
print('-' * 47)
for modelo, maes in mae_folds.items():
    print(f'{modelo:<20} {np.mean(maes):>10.4f} {np.std(maes):>15.4f}')

## 7. Teste Estatístico entre Modelos (Wilcoxon)

Diferenças pequenas de MAE entre modelos podem ser ruído — não necessariamente o modelo A é melhor que B. O **teste de Wilcoxon** (não-paramétrico, adequado para amostras pequenas) verifica se a diferença é estatisticamente significativa.

- **H0:** Os dois modelos têm desempenho equivalente
- **H1:** Um modelo é superior ao outro
- Se **p < 0.05** → rejeitamos H0 → diferença significativa

In [ ]:
erros_lr  = np.abs(np.array(mae_folds['Regressão Linear']))
erros_rf  = np.abs(np.array(mae_folds['Random Forest']))
erros_xgb = np.abs(np.array(mae_folds['XGBoost']))

comparacoes = [
    ('Regressão Linear', 'Random Forest', erros_lr,  erros_rf),
    ('Regressão Linear', 'XGBoost',       erros_lr,  erros_xgb),
    ('Random Forest',    'XGBoost',       erros_rf,  erros_xgb),
]

print('Teste de Wilcoxon entre modelos:')
print(f'{"Comparação":<40} {"p-value":>10} {"Significativo?":>15}')
print('-' * 67)
for m1, m2, e1, e2 in comparacoes:
    try:
        stat, p = wilcoxon(e1, e2)
        sig = 'Sim (p<0.05)' if p < 0.05 else 'Não'
        print(f'{m1} vs {m2:<20} {p:>10.4f} {sig:>15}')
    except Exception as ex:
        print(f'{m1} vs {m2}: {ex}')

## 8. Importância de Features

O Random Forest e o XGBoost calculam automaticamente a **importância de cada feature** — o quanto cada variável contribui para reduzir o erro do modelo.

Isso responde perguntas como: *o ciclo de 4 anos importa mais que a forma recente?*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (modelo, nome) in zip(axes, [(rf, 'Random Forest'), (xgb, 'XGBoost')]):
    importancias = pd.Series(modelo.feature_importances_, index=features).sort_values()
    importancias.plot(kind='barh', ax=ax, color='steelblue', edgecolor='black')
    ax.set_title(f'Importância de Features — {nome}')
    ax.set_xlabel('Importância')

plt.tight_layout()
plt.savefig('../article/figures/importancia_features.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nImportância — Random Forest:')
print(pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False).round(4))

## 9. Análise de Erros

Identificamos quais seleções o modelo mais errou na Copa 2022 — útil para entender as limitações e casos onde o modelo falha sistematicamente.

In [ ]:
df_2022 = df[df['copa_alvo'] == 2022].copy().reset_index(drop=True)

df_2022['pred_lr']  = y_pred_lr
df_2022['pred_rf']  = y_pred_rf
df_2022['pred_xgb'] = y_pred_xgb

df_2022['erro_lr']  = abs(df_2022['media_gols_copa'] - df_2022['pred_lr'])
df_2022['erro_rf']  = abs(df_2022['media_gols_copa'] - df_2022['pred_rf'])
df_2022['erro_xgb'] = abs(df_2022['media_gols_copa'] - df_2022['pred_xgb'])

print('Top 10 maiores erros — Random Forest (Copa 2022):')
print(df_2022[['selecao', 'media_gols_copa', 'pred_rf', 'erro_rf']]
      .sort_values('erro_rf', ascending=False)
      .head(10)
      .to_string(index=False))

In [ ]:
# Visualização dos erros por seleção
df_erros = df_2022[['selecao', 'erro_lr', 'erro_rf', 'erro_xgb']].set_index('selecao')
df_erros = df_erros.sort_values('erro_rf', ascending=False).head(15)

df_erros.plot(kind='bar', figsize=(14, 5), color=['steelblue', 'coral', 'green'],
              edgecolor='black', alpha=0.8)
plt.title('Erro Absoluto por Seleção — Copa 2022 (Top 15)')
plt.ylabel('Erro Absoluto (gols)')
plt.xlabel('')
plt.xticks(rotation=45, ha='right')
plt.legend(['Regressão Linear', 'Random Forest', 'XGBoost'])
plt.tight_layout()
plt.savefig('../article/figures/erros_por_selecao.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Resumo dos Resultados

### Métricas no Conjunto de Teste (Copa 2022)

In [ ]:
resumo = pd.DataFrame({
    'Modelo': ['Regressão Linear', 'Random Forest', 'XGBoost'],
    'MAE Teste':  [mae_lr, mae_rf, mae_xgb],
    'RMSE Teste': [rmse_lr, rmse_rf, rmse_xgb],
    'MAE CV (média)': [np.mean(mae_folds['Regressão Linear']),
                       np.mean(mae_folds['Random Forest']),
                       np.mean(mae_folds['XGBoost'])],
    'MAE CV (std)': [np.std(mae_folds['Regressão Linear']),
                     np.std(mae_folds['Random Forest']),
                     np.std(mae_folds['XGBoost'])]
}).round(4)

print('Tabela Comparativa Final:')
print(resumo.to_string(index=False))

# Salvar para o artigo
resumo.to_csv('../article/tables/comparacao_modelos.csv', index=False)
print('\nTabela salva em article/tables/comparacao_modelos.csv')

## 11. Conclusões do Notebook

**Melhor modelo:** Random Forest (menor MAE no teste — 0.4512 gols por jogo)

**Principais achados:**
- Um erro médio de ~0.45 gols por jogo é razoável para um problema inerentemente imprevisível como futebol
- A Regressão Linear apresenta RMSE menor — comete menos erros grandes, mas é mais conservadora
- O XGBoost ficou em último — o dataset pequeno (216 amostras de treino) não favorece modelos complexos
- O teste de Wilcoxon deve ser analisado para confirmar se as diferenças são estatisticamente significativas
- Todos os modelos têm dificuldade com seleções nos extremos (muito fracas ou muito fortes)

**Limitações identificadas:**
- Amistosos e jogos competitivos têm o mesmo peso nas features
- O nível do adversário não é considerado (gol contra San Marino vale o mesmo que contra Alemanha)
- Mudanças de treinador ou geração de jogadores não são capturadas

**Próximo passo:** `04_previsao_2026.ipynb` — previsão da média de gols para a Copa 2026.